# Well Performance Analytics Dashboard

Comprehensive analysis of oil and gas well production, efficiency, and performance metrics.

---

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import requests
import json

# Set style for visualizations
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# API Configuration
BASE_URL = 'http://localhost:8000/api/v1'

print('Setup complete. Ready to load data.')

## 2. Load Data from API

In [ ]:
# Load wells data
wells_response = requests.get(f'{BASE_URL}/wells')
wells_data = wells_response.json()
wells_df = pd.DataFrame(wells_data)

print(f'Loaded {len(wells_df)} wells')
print('\nWells Summary:')
print(wells_df[['id', 'well_name', 'field_name', 'well_type', 'status']].head(10))

In [ ]:
# Load production trend data
trend_response = requests.get(f'{BASE_URL}/analytics/production-trend')
trend_data = trend_response.json()
trend_df = pd.DataFrame(trend_data)
trend_df['log_date'] = pd.to_datetime(trend_df['log_date'])

print(f'Loaded {len(trend_df)} production log records')
print('\nProduction Trend Summary:')
print(trend_df.describe())

In [ ]:
# Load dashboard KPIs
kpi_response = requests.get(f'{BASE_URL}/analytics/dashboard-kpi')
dashboard_kpi = kpi_response.json()

print('Portfolio-level KPIs:')
for key, value in dashboard_kpi.items():
    print(f'  {key}: {value}')

## 3. Portfolio-Level Analysis

In [ ]:
# Well status distribution
status_counts = wells_df['status'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Status pie chart
status_counts.plot(kind='pie', ax=ax1, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6'])
ax1.set_title('Well Status Distribution', fontsize=14, fontweight='bold')
ax1.set_ylabel('')

# Well type distribution
type_counts = wells_df['well_type'].value_counts()
type_counts.plot(kind='bar', ax=ax2, color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
ax2.set_title('Well Type Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Well Type')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nWell Status Summary:')
print(status_counts)
print('\nWell Type Summary:')
print(type_counts)

In [ ]:
# Production trend over time by field
daily_production = trend_df.groupby(['log_date', 'field_name']).agg({
    'total_oil_bbl': 'sum',
    'total_gas_mcf': 'sum',
    'total_water_bbl': 'sum'
}).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))

for field in daily_production['field_name'].unique():
    field_data = daily_production[daily_production['field_name'] == field].sort_values('log_date')
    ax.plot(field_data['log_date'], field_data['total_oil_bbl'], label=field, marker='o', markersize=3, linewidth=2)

ax.set_xlabel('Date')
ax.set_ylabel('Total Oil Production (BBL)')
ax.set_title('Oil Production Trend by Field', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Well-Level Performance Analysis

In [ ]:
# Get top 5 producing wells
top_producers_response = requests.get(f'{BASE_URL}/analytics/top-producers')
top_producers = pd.DataFrame(top_producers_response.json())

print('Top 5 Producing Wells:')
print(top_producers[['well_name', 'field_name', 'total_oil_bbl', 'avg_daily_oil']].to_string(index=False))

# Visualize top producers
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_producers['well_name'], top_producers['total_oil_bbl'], color='#3498db')
ax.set_xlabel('Total Oil Production (BBL)')
ax.set_title('Top 5 Producing Wells', fontsize=14, fontweight='bold')

# Add value labels on bars
for i, (well, value) in enumerate(zip(top_producers['well_name'], top_producers['total_oil_bbl'])):
    ax.text(value, i, f' {value:,.0f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Load downtime data and analyze
downtime_response = requests.get(f'{BASE_URL}/analytics/downtime-summary')
downtime_df = pd.DataFrame(downtime_response.json())

print('Top 10 Wells by Downtime:')
print(downtime_df.nlargest(10, 'total_downtime_hrs')[['well_name', 'field_name', 'total_downtime_hrs']].to_string(index=False))

# Visualize downtime
fig, ax = plt.subplots(figsize=(12, 6))
top_downtime = downtime_df.nlargest(10, 'total_downtime_hrs')
bars = ax.barh(top_downtime['well_name'], top_downtime['total_downtime_hrs'], color='#e74c3c')
ax.set_xlabel('Total Downtime (Hours)')
ax.set_title('Top 10 Wells by Downtime', fontsize=14, fontweight='bold')

# Add value labels
for i, (well, value) in enumerate(zip(top_downtime['well_name'], top_downtime['total_downtime_hrs'])):
    ax.text(value, i, f' {value:,.0f}h', va='center')

plt.tight_layout()
plt.show()

## 5. Efficiency Analysis

In [ ]:
# Calculate efficiency metrics for top wells
top_well_ids = top_producers['well_id'].head(5).tolist()
efficiency_data = []

for well_id in top_well_ids:
    try:
        response = requests.get(f'{BASE_URL}/analytics/efficiency-metrics/{well_id}')
        data = response.json()
        efficiency_data.append(data)
    except:
        pass

efficiency_df = pd.DataFrame(efficiency_data)

print('Efficiency Metrics for Top 5 Producing Wells:')
print(efficiency_df[['well_name', 'uptime_percentage', 'downtime_percentage', 'total_operational_days']].to_string(index=False))

In [ ]:
# Visualize uptime vs downtime
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Uptime bar chart
colors = ['#2ecc71' if x > 80 else '#f39c12' if x > 60 else '#e74c3c' for x in efficiency_df['uptime_percentage']]
ax1.bar(efficiency_df['well_name'], efficiency_df['uptime_percentage'], color=colors)
ax1.axhline(y=80, color='g', linestyle='--', alpha=0.5, label='Target (80%)')
ax1.set_ylabel('Uptime Percentage (%)')
ax1.set_title('Well Uptime Percentage', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 105)
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# Operational days comparison
ax2.bar(efficiency_df['well_name'], efficiency_df['total_operational_days'], color='#3498db')
ax2.set_ylabel('Operational Days')
ax2.set_title('Total Operational Days', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Production Ranking and Comparison

In [ ]:
# Get production ranking by well
ranking_response = requests.get(f'{BASE_URL}/analytics/production-ranking?group_by=well')
ranking_df = pd.DataFrame(ranking_response.json())

print('Production Ranking (Top 10 Wells):')
print(ranking_df.head(10)[['ranking', 'name', 'production_oil_bbl', 'production_gas_mcf']].to_string(index=False))

# Visualize ranking
top_10_ranking = ranking_df.head(10)
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(top_10_ranking)), top_10_ranking['production_oil_bbl'].values, color='#3498db')
ax.set_yticks(range(len(top_10_ranking)))
ax.set_yticklabels(top_10_ranking['name'].values)
ax.invert_yaxis()
ax.set_xlabel('Total Oil Production (BBL)')
ax.set_title('Production Ranking - Top 10 Wells', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Field-level comparison
field_comparison_response = requests.get(f'{BASE_URL}/analytics/field-comparison')
field_comparison = pd.DataFrame(field_comparison_response.json())

print('Field-Level Comparison:')
print(field_comparison.to_string(index=False))

# Visualize field comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Oil production by field
ax1.bar(field_comparison['field_name'], field_comparison['total_oil_bbl'], color='#3498db')
ax1.set_ylabel('Total Oil Production (BBL)')
ax1.set_title('Oil Production by Field', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Gas production by field
ax2.bar(field_comparison['field_name'], field_comparison['total_gas_mcf'], color='#2ecc71')
ax2.set_ylabel('Total Gas Production (MCF)')
ax2.set_title('Gas Production by Field', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Statistical Summary

In [ ]:
# Production statistics
print('\n=== PRODUCTION STATISTICS ===')
print(f'\nTotal Oil Production: {trend_df["total_oil_bbl"].sum():,.0f} BBL')
print(f'Total Gas Production: {trend_df["total_gas_mcf"].sum():,.0f} MCF')
print(f'Total Water Produced: {trend_df["total_water_bbl"].sum():,.0f} BBL')
print(f'\nAverage Daily Oil: {trend_df.groupby("log_date")["total_oil_bbl"].sum().mean():,.0f} BBL/day')
print(f'Average Daily Gas: {trend_df.groupby("log_date")["total_gas_mcf"].sum().mean():,.0f} MCF/day')
print(f'\nPeak Daily Oil Production: {trend_df.groupby("log_date")["total_oil_bbl"].sum().max():,.0f} BBL')
print(f'Peak Daily Gas Production: {trend_df.groupby("log_date")["total_gas_mcf"].sum().max():,.0f} MCF')

In [ ]:
# Portfolio statistics
print('\n=== PORTFOLIO STATISTICS ===')
print(f'Total Wells: {dashboard_kpi["total_wells"]}')
print(f'Active Wells: {dashboard_kpi["active_wells"]}')
print(f'Portfolio Uptime: {dashboard_kpi["avg_portfolio_uptime_pct"]:.2f}%')
print(f'Total Maintenance Cost: ${dashboard_kpi["total_maintenance_cost_usd"]:,.2f}')
print(f'Average Maintenance Hours: {dashboard_kpi["avg_maintenance_hrs"]:.2f} hours')

In [ ]:
# Correlation analysis
print('\n=== CORRELATION ANALYSIS ===')
correlation_data = trend_df[['total_oil_bbl', 'total_gas_mcf', 'total_water_bbl']].corr()
print('\nCorrelation Matrix:')
print(correlation_data)

# Visualize correlation
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlation_data, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Production Metrics Correlation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Summary and Insights

In [ ]:
print("\n" + "="*60)
print("ANALYTICS SUMMARY")
print("="*60)

print("\n📊 PORTFOLIO HEALTH:")
active_pct = (dashboard_kpi['active_wells'] / dashboard_kpi['total_wells']) * 100
print(f"  • Active Wells: {dashboard_kpi['active_wells']}/{dashboard_kpi['total_wells']} ({active_pct:.1f}%)")
print(f"  • Portfolio Uptime: {dashboard_kpi['avg_portfolio_uptime_pct']:.2f}%")

print("\n📈 PRODUCTION METRICS:")
print(f"  • Total Oil Production: {trend_df['total_oil_bbl'].sum():,.0f} BBL")
print(f"  • Total Gas Production: {trend_df['total_gas_mcf'].sum():,.0f} MCF")
print(f"  • Average Daily Oil: {trend_df.groupby('log_date')['total_oil_bbl'].sum().mean():,.0f} BBL/day")

print("\n💰 MAINTENANCE & COSTS:")
print(f"  • Total Maintenance Cost: ${dashboard_kpi['total_maintenance_cost_usd']:,.2f}")
print(f"  • Average Maintenance Per Event: {dashboard_kpi['avg_maintenance_hrs']:.2f} hours")

print("\n🎯 TOP PERFORMER:")
top_well = top_producers.iloc[0]
print(f"  • Well: {top_well['well_name']} ({top_well['field_name']})")
print(f"  • Total Production: {top_well['total_oil_bbl']:,.0f} BBL")
print(f"  • Average Daily: {top_well['avg_daily_oil']:,.0f} BBL/day")

print("\n⚠️  ATTENTION REQUIRED:")
worst_uptime = efficiency_df.nsmallest(1, 'uptime_percentage')
if len(worst_uptime) > 0:
    print(f"  • {worst_uptime.iloc[0]['well_name']}: {worst_uptime.iloc[0]['uptime_percentage']:.2f}% uptime")

print("\n" + "="*60)